# Nutrition5k — Experiment 4: Volume Scalar Pipeline

Two-stage: (1) train mass regressor, (2) combine with Exp 1 per-gram model for full nutrition prediction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys

if os.path.isdir('/content/Nutrition5k'):
    subprocess.run(['git', '-C', '/content/Nutrition5k', 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/T0MYYY/Nutrition5k.git', '/content/Nutrition5k', '-b', 'master', '-q'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/Nutrition5k', '-q'], check=True)

if '/content/Nutrition5k' not in sys.path:
    sys.path.insert(0, '/content/Nutrition5k')
import importlib; importlib.invalidate_caches()

CONFIG_DIR = '/content/Nutrition5k/configs/colab'
print(f'Ready. Config: {CONFIG_DIR}')

## GPU Check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Prerequisites

Exp 1 must already be trained. Verify the checkpoint exists:

In [ ]:
import os

exp1_ckpt = '/content/drive/MyDrive/nutrition5k/checkpoints/exp1/best.pt'
vol_csv   = '/content/drive/MyDrive/nutrition5k/data/metadata/volume_estimates.csv'

assert os.path.isfile(exp1_ckpt), f'Exp 1 checkpoint not found: {exp1_ckpt}\n→ Run exp1_portion_independent.ipynb first'
assert os.path.isfile(vol_csv),   f'volume_estimates.csv not found: {vol_csv}\n→ Run 00_prepare_data.ipynb first'

print('Exp 1 checkpoint found')
print(f'volume_estimates.csv found ({os.path.getsize(vol_csv):,} bytes)')

## Load data into RAM

First run: parallel Drive read + zip to Drive. Next runs: zip→RAM in seconds.

In [ ]:
import os, subprocess, threading

subprocess.run(['apt-get', 'install', '-qq', '-y', 'zstd'],
               check=True, capture_output=True)

DRIVE  = '/content/drive/MyDrive/nutrition5k/data'
DST    = '/dev/shm'

def _extract(archive, marker_dir, min_dishes):
    if os.path.isdir(marker_dir) and sum(1 for d in os.scandir(marker_dir) if d.is_dir()) > min_dishes:
        print(f'  {archive}: already extracted'); return
    src = f'{DRIVE}/{archive}'
    print(f'  Extracting {archive} ...')
    subprocess.run(f'zstd -d --stdout "{src}" | tar -x -C {DST}',
                   shell=True, check=True)
    print(f'  Done: {archive}')

os.makedirs('/dev/shm/realsense_overhead', exist_ok=True)
os.makedirs('/dev/shm/side_angles', exist_ok=True)

# depth archives for training; rgb_test for Stage 2 cal/g evaluation
todos = [
    ('depth_train.tar.zst', '/dev/shm/realsense_overhead', 50),
    ('depth_test.tar.zst',  '/dev/shm/realsense_overhead', 50),
    ('rgb_test.tar.zst',    '/dev/shm/side_angles',        0),
]
threads = [threading.Thread(target=_extract, args=t, daemon=True) for t in todos]
for t in threads: t.start()
for t in threads: t.join()

n_depth = sum(1 for d in os.scandir('/dev/shm/realsense_overhead') if d.is_dir())
n_rgb   = sum(1 for d in os.scandir('/dev/shm/side_angles') if d.is_dir())
print(f'Ready: {n_depth} overhead dishes, {n_rgb} side-angle dishes in /dev/shm')

## Stage 1: Train Mass Regressor

In [ ]:
import torch, yaml

cfg_path = f'{CONFIG_DIR}/exp4_volume_scalar.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
    if vram_gb >= 80:   bs = 256
    elif vram_gb >= 40: bs = 128
    elif vram_gb >= 24: bs = 64
    else:               bs = 32
    scale = bs / 256
    cfg['training']['batch_size'] = bs
    cfg['training']['lr']      = round(3e-4 * scale, 7)
    cfg['training']['head_lr'] = round(1e-3 * scale, 7)

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)
print(f"batch={cfg['training']['batch_size']}  lr={cfg['training']['lr']:.2e}  head_lr={cfg['training']['head_lr']:.2e}")

In [ ]:
from nutrition5k_pkg.train import run
best_mass_ckpt = run(f'{CONFIG_DIR}/exp4_volume_scalar.yaml')
print('Mass regressor checkpoint:', best_mass_ckpt)

## Stage 2: Combined Evaluation (mass regressor × per-gram model)

For each test dish:
- overhead RGB + volume → mass regressor → predicted mass
- side-angle frame → per-gram model (Exp 1) → predicted cal/g
- total calories = mass × cal/g

In [ ]:
import torch, numpy as np, csv as _csv, os
from PIL import Image
from nutrition5k_pkg.train import load_config, _build_model
from nutrition5k_pkg.data.metadata import load_dish_metadata, load_official_split
from nutrition5k_pkg.data.transforms import get_val_transform
from nutrition5k_pkg.models.multitask_head import MultitaskNutritionNet
from nutrition5k_pkg.metrics import compute_mae_report, print_results_table

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
overhead_tfm = get_val_transform(pre_resized=False)  # overhead images not pre-resized
side_tfm     = get_val_transform(pre_resized=True)   # side-angle frames are pre-resized

# Load per-gram model (Exp 1)
pg_tasks = ['cal_per_g', 'fat_per_g', 'carb_per_g', 'protein_per_g']
pg_model = MultitaskNutritionNet(tasks=pg_tasks).to(device)
pg_model.load_state_dict(torch.load('/content/drive/MyDrive/nutrition5k/checkpoints/exp1/best.pt', map_location=device))
pg_model.eval()

# Load mass regressor (Exp 4)
exp4_cfg = load_config(f'{CONFIG_DIR}/exp4_volume_scalar.yaml')
mass_model = _build_model(exp4_cfg).to(device)
mass_model.load_state_dict(torch.load('/content/drive/MyDrive/nutrition5k/checkpoints/exp4/best.pt', map_location=device))
mass_model.eval()

# Official test split
_, _, test_ids = load_official_split(
    exp4_cfg['data']['train_ids_txt'],
    exp4_cfg['data']['test_ids_txt'],
)

# Metadata + volume map
meta = load_dish_metadata(exp4_cfg['data']['cafe1_csv'], exp4_cfg['data'].get('cafe2_csv'))
vol_path = '/content/drive/MyDrive/nutrition5k/data/metadata/volume_estimates.csv'
with open(vol_path) as f:
    vol_map = {row['dish_id']: float(row['volume_cm3']) for row in _csv.DictReader(f)}

OVERHEAD = '/dev/shm/realsense_overhead'
SIDE     = '/dev/shm/side_angles'

preds_cal, gt_cal = [], []
with torch.no_grad():
    for dish_id in test_ids:
        rgb_path = os.path.join(OVERHEAD, dish_id, 'rgb.png')
        if not os.path.isfile(rgb_path) or dish_id not in vol_map:
            continue
        overhead_img = overhead_tfm(Image.open(rgb_path).convert('RGB')).unsqueeze(0).to(device)
        vol_t = torch.tensor([[vol_map[dish_id]]], dtype=torch.float32).to(device)
        mass_pred = mass_model(overhead_img, vol_t).item()

        frames_dir = os.path.join(SIDE, dish_id, 'frames')
        if not os.path.isdir(frames_dir):
            continue
        frames = sorted(f for f in os.listdir(frames_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        if not frames:
            continue
        side_img = side_tfm(Image.open(os.path.join(frames_dir, frames[0])).convert('RGB')).unsqueeze(0).to(device)
        cal_per_g = pg_model(side_img)['cal_per_g'].item()

        preds_cal.append(mass_pred * cal_per_g)
        gt_cal.append(meta[dish_id]['calories'])

print(f'Evaluated {len(preds_cal)} dishes')
results = {'calories': compute_mae_report(np.array(preds_cal), np.array(gt_cal))}
print_results_table(results)

results_dir = '/content/drive/MyDrive/nutrition5k/results/exp4'
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, 'predictions.csv'), 'w', newline='') as f:
    w = _csv.writer(f)
    w.writerow(['pred_calories', 'gt_calories'])
    for p, g in zip(preds_cal, gt_cal):
        w.writerow([p, g])
print('Results saved to:', results_dir)

In [ ]:
from google.colab import runtime
runtime.unassign()